In [75]:
import random
import numpy as np


def to_bits(x,n):
    return np.array([(x>>i) & 1 for i in range(n)],dtype=np.uint8)

def to_int(v):
    x=0
    for i,b in enumerate(v):
        x|=(int(b)&1)<<i
        return x
    
def rndm_bi_mat(n):
    return np.random.randint(0, 2, size=(n, n), dtype=np.uint8)


def gf2_rank(M):
    M=M.copy().astype(np.uint8)
    rs,cs = M.shape
    rk = 0
    col = 0
    for r in range(rs):
        pivot=None
        for i in range(r,rs):
            if M[i,col]==1:
                pivot = i
                break
        if pivot is None:
            col+=1
            if col >= cs:
                break
            r-=1
            continue
        if pivot != r:
            M[[r,pivot]] = M[[pivot,r]]
        for i in range(r+1,rs):
            if M[i,col]==1:
                M[i,:]^=M[r,:]
        rk +=1
        col +=1
        if col >= cs:
            break
    return rk

def rng_inv_mat(n):
    while True:
        M = rndm_bi_mat(n)
        if gf2_rank(M) == n:
            return M
        
def apply_lin(M,x,n):
    v=to_bits(x,n)
    y=M.dot(v)%2
    return to_int(y)

def rng_bit_perm(n):
    perm = list(range(n))
    random.shuffle(perm)
    return perm

def apply_perm(perm,x,n):
    v=to_bits(x,n)
    w = np.zeros_like(v)
    for new, old in enumerate(perm):
        w[new]=v[old]
    return to_int(w)

def gen_le_pair(n):
    N = 1<<n

    g1 = list(range(N))
    random.shuffle(g1)

    s= rng_inv_mat(n)

    p = rng_bit_perm(n)

    g2 = [0]*N
    for x in range(N):
        x_perm = apply_perm(p,x,n)
        y1=g1[x_perm]
        y2 = apply_lin(s,y1,n)
        g2[x]=y2 
    return (g1,g2,s,p)

if __name__ == "__main__":
    n=5
    g1,g2,s,p = gen_le_pair(n)
    print("g1:",g1)
    print("g2:",g2)
    print("s:\n",s)
    print("p:",p)

g1: [17, 23, 11, 9, 27, 25, 29, 5, 10, 21, 8, 1, 2, 12, 24, 30, 14, 7, 13, 16, 28, 4, 18, 15, 6, 19, 22, 0, 26, 3, 20, 31]
g2: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
s:
 [[1 0 0 1 1]
 [0 1 1 1 0]
 [0 0 1 0 0]
 [1 1 0 0 0]
 [1 0 0 0 0]]
p: [4, 3, 1, 2, 0]


In [119]:
import random
import numpy as np

# ---------- bit utilities ----------

def int_to_bits(x, n):
    """
    Map integer x in [0, 2^n) to an n-dimensional vector over GF(2),
    using little-endian bit order: v[0] = LSB.
    """
    return np.array([(x >> i) & 1 for i in range(n)], dtype=np.uint8)

def bits_to_int(v):
    """
    Inverse of int_to_bits under the same convention.
    """
    x = 0
    for i, b in enumerate(v):
        x |= (int(b) & 1) << i
    return x

# ---------- linear maps over GF(2) ----------

def random_binary_matrix(n):
    """
    Random n x n matrix over GF(2).
    """
    return np.random.randint(0, 2, size=(n, n), dtype=np.uint8)

def gf2_rank(M):
    """
    Compute rank of M over GF(2) via Gaussian elimination.
    """
    M = M.copy().astype(np.uint8)
    rows, cols = M.shape
    rank = 0
    col = 0
    for r in range(rows):
        if col >= cols:
            break

        # find a pivot row with 1 in column 'col'
        pivot = None
        for i in range(r, rows):
            if M[i, col] == 1:
                pivot = i
                break

        if pivot is None:
            # no pivot in this column; move to next column
            col += 1
            # stay at same r (outer loop will ++r, so undo here)
            r -= 1
            continue

        # swap pivot row into position r
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]

        # eliminate below
        for i in range(r + 1, rows):
            if M[i, col] == 1:
                M[i, :] ^= M[r, :]

        rank += 1
        col += 1

    return rank

def random_invertible_matrix(n):
    """
    Sample uniformly from M_n(F_2) until we get a full-rank matrix,
    i.e. an element of GL(n,2).
    """
    while True:
        M = random_binary_matrix(n)
        if gf2_rank(M) == n:
            return M

def apply_linear(M, x, n):
    """
    Apply linear map represented by matrix M to the n-bit vector x.
    """
    v = int_to_bits(x, n)
    y = M.dot(v) % 2
    return bits_to_int(y)

# ---------- bit permutation ----------

def random_bit_perm(n):
    """
    Random permutation of bit positions {0,...,n-1}.
    """
    perm = list(range(n))
    random.shuffle(perm)
    return perm

def apply_bit_perm(perm, x, n):
    """
    Apply bit permutation 'perm' to n-bit integer x.
    perm[new_pos] = old_pos.
    """
    v = int_to_bits(x, n)
    w = np.zeros_like(v)
    for new_pos, old_pos in enumerate(perm):
        w[new_pos] = v[old_pos]
    return bits_to_int(w)

# ---------- generate linearly equivalent pair ----------

def generate_le_pair(n):
    """
    Generate:
      - g1: random permutation on {0,...,2^n-1}
      - g2: permutation linearly equivalent to g1 via S and P
      - S: random invertible n x n matrix over GF(2)
      - P: random bit permutation of {0,...,n-1}
    so that  g2[x] = S( g1( P(x) ) )  for all x.
    """
    N = 1 << n  # 2^n

    # 1) random base permutation g1
    g1 = list(range(N))
    random.shuffle(g1)

    # 2) random invertible output map S
    S = random_invertible_matrix(n)

    # 3) random input bit-permutation P
    P = random_bit_perm(n)

    # 4) define g2(x) = S( g1( P(x) ) )
    g2 = [0] * N
    for x in range(N):
        x_p = apply_bit_perm(P, x, n)    # P(x)
        y1  = g1[x_p]                    # g1(P(x))
        y2  = apply_linear(S, y1, n)     # S(g1(P(x)))
        g2[x] = y2

    # sanity checks
    assert len(set(g1)) == N, "g1 is not a permutation!"
    assert len(set(g2)) == N, "g2 is not a permutation!"
    assert gf2_rank(S) == n,  "S is not invertible!"
    assert sorted(P) == list(range(n)), "P is not a permutation of bit positions!"

    # verify the LE relation explicitly
    for x in range(N):
        lhs = g2[x]
        rhs = apply_linear(S, g1[apply_bit_perm(P, x, n)], n)
        assert lhs == rhs, "LE relation failed at x = {}".format(x)

    return g1, g2, S, P

if __name__ == "__main__":
    n = 5
    g1, g2, S, P = generate_le_pair(n)
    print("g1:", g1)
    print("g2:", g2)
    print("S:\n", S)
    print("P:", P)


g1: [3, 31, 11, 20, 30, 13, 23, 10, 9, 6, 5, 24, 21, 4, 8, 14, 25, 17, 15, 0, 29, 12, 19, 18, 16, 26, 27, 7, 28, 2, 22, 1]
g2: [12, 11, 1, 10, 5, 17, 22, 15, 28, 29, 8, 13, 25, 3, 0, 31, 30, 27, 18, 16, 21, 9, 6, 23, 24, 19, 26, 14, 7, 20, 4, 2]
S:
 [[0 0 1 1 0]
 [1 1 1 0 1]
 [0 1 0 0 0]
 [0 1 0 1 1]
 [0 0 1 0 0]]
P: [3, 2, 4, 0, 1]
